In [ ]:
!pip install -e /content/drive/MyDrive/kinya-assistant/Inference/


In [ ]:
!pip install  Cython

In [ ]:
%cd /content/drive/MyDrive/kinya-assistant/Inference/monotonic_align

# Create the monotonic_align directory if it doesn't exist
# This directory will contain the compiled C extension
!mkdir -p monotonic_align

# Compile the C extension using Cython
# The --inplace flag builds the extension in the source directory
# This is required for the duration alignment algorithm in the TTS system
!python setup.py build_ext --inplace

In [ ]:
%cd /content/drive/MyDrive/kinya-assistant/Inference/monotonic_align

# Install the monotonic_align package in development mode
# This allows importing the module after compilation
# The -e flag (editable mode) enables changes without reinstallation
!pip install -e .

In [ ]:
import monotonic_align
print("Successfully imported!")

In [ ]:
!apt-get update && apt-get install -y sox libsox-fmt-all


In [ ]:
!pip install nemo-toolkit


In [ ]:
%pip install --no-cache-dir -r /content/drive/MyDrive/kinya-assistant/stt/requirements.txt


In [ ]:
!pip install lightning
!pip install lhotse
!pip install jiwer
!pip install pyannote

In [ ]:
from huggingface_hub import login
from google.colab import userdata

hf_token= userdata.get("HF_TOKEN")

login(token=hf_token, add_to_git_credential=True)


In [ ]:
from kinyatts.tts.commons import intersperse
from kinyatts.tts.utils import get_hparams_from_file, load_checkpoint
from kinyatts.tts.models import SynthesizerTrn
from kinyatts.tts.text import text_to_sequence
from kinyatts.tts.text.symbols import symbols

import os
import nemo.collections.asr as nemo_asr
# from pydub import AudioSegment
# import pyaudioconvert as pac
import numpy as np
import time as time
import uuid

#import the model using hugging face
hf_model = nemo_asr.models.EncDecRNNTBPEModel.from_pretrained(model_name="mbazaNLP/Kinyarwanda_nemo_stt_conformer_model")


In [ ]:
!pip install gradio
!pip install transformers
!pip install torchaudio
!pip install TTS
!pip install "numpy<2.1.0,>1.26.0"

In [ ]:
!pip uninstall transformers
!pip install numpy==1.24.3 --force-reinstall


In [ ]:
!pip install -q openai-whisper


In [ ]:

# Import libraries
import gradio as gr
import torchaudio
import torch
import whisper
from TTS.api import TTS

# Load Whisper model (for Speech Recognition)
whisper_model = whisper.load_model("small")  # 'tiny', 'base', 'small', 'medium', 'large'


# tts_model = TTS(model_name="tts_models/en/ljspeech/tacotron2-DDC")

# Function to transcribe audio and convert text to speech (TTS)
def transcribe_audio(audio_path):
    # Load and preprocess audio
    waveform, sample_rate = torchaudio.load(audio_path)

    # Resample if needed
    if sample_rate != 16000:
        resampler = torchaudio.transforms.Resample(orig_freq=sample_rate, new_freq=16000)
        waveform = resampler(waveform)

    # Save processed waveform to a temp file because Whisper expects a file path
    temp_audio_path = "temp.wav"
    torchaudio.save(temp_audio_path, waveform, 16000)

    # Transcribe using Whisper
    result = whisper_model.transcribe(temp_audio_path)
    transcription = result["text"]

    # Basic Kinyarwanda QA dictionary
    qa_dict = {
        "amakuru yawe": "Nari meza, murakoze kubaza!",
        "wiriwe": "Nari meza, urakomeye?",
        "amafaranga angahe": "Mfite amafaranga 1000 RWF",
    }

    # Match transcription to known questions
    response = qa_dict.get(transcription.lower(), "Ndumva ntazi icyo uvuga.")

    # Text-to-Speech: save response as audio
    tts_filename = "response.wav"
    hf_model.tts_to_file(text=response, file_path=tts_filename)

    return transcription, tts_filename


# Text-to-Speech function
def text_to_speech(text, speaker_wav, language):
    file_path = "text_response.wav"
    hf_model.tts_to_file(
        text=text,
        file_path=file_path,
        speaker_wav=speaker_wav,
        language=language
    )
    return file_path


# Gradio Interface for Voice Assistant
voice_assistant = gr.Interface(
    fn=transcribe_audio,
    inputs=gr.Audio(type="filepath"),
    outputs=[gr.Textbox(label="Transcription"), gr.Audio(label="Response Audio")],
    live=False,
    title="Kinyarwanda Voice Assistant"
)

# Gradio Interface for Text-to-Speech
text_to_speech_interface = gr.Interface(
    fn=text_to_speech,
    inputs=[
        gr.Textbox(label="Enter your text"),
        gr.Textbox(label="Path to target speaker WAV file", value="/content/speaker.wav"),
        gr.Dropdown(label="Language", choices=["rw"], value="rw")
    ],
    outputs=gr.Audio(label="Generated Speech"),
    title="Voice Synthesis with TTS",
    description="Input text to synthesize speech using a target voice and language."
)

# Combine interfaces into tabs
app = gr.TabbedInterface(
    interface_list=[voice_assistant, text_to_speech_interface],
    tab_names=["Voice Assistant", "Text-to-Speech"]
)

# Launch the app
app.launch(share=True, debug=True)
